# Module 3 · Solutions
Attempt first. Each solution notes the *decision*, not just the code — in data work the decision is the skill.

In [ ]:
import pandas as pd, numpy as np
import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
prices = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"])

## 3A

In [ ]:
# Ex1 - close > 17000 in 2025
mask = (prices["close"] > 17000) & (prices["date"].dt.year == 2025)
print("Rows:", mask.sum())

# Ex2 - best 5 days
prices["return_pct"] = prices["close"].pct_change() * 100
print(prices.sort_values("return_pct", ascending=False).head(5)[["date","return_pct"]])

# Ex3 - average range per year (choppiest year = regime change confirmation)
prices["range"] = prices["high"] - prices["low"]
by_year = prices.groupby(prices["date"].dt.year)["range"].mean().round(1)
print(by_year)
print("Choppiest:", by_year.idxmax(), "- matches the registry's regime shift (and 2024 contains the fat-finger high, which inflates range; drop 2024-08-13 and recheck!)")

## 3B

In [ ]:
# Ex1 - city totals, clean vs raw
raw = pd.read_csv(BASE + "messy_transactions.csv")

txn = raw.drop_duplicates()
txn = txn[txn["amount_inr"] > 0].copy()
usd = txn["amount_inr"] < 20
txn.loc[usd, "amount_inr"] *= 83.0
txn["city"] = txn["city"].str.strip().str.title()

clean_city = txn.groupby("city")["amount_inr"].sum().sort_values(ascending=False)
raw_city   = raw.groupby("city")["amount_inr"].sum().sort_values(ascending=False)
print("CLEAN top city:", clean_city.index[0], f"Rs {clean_city.iloc[0]:,.0f}")
print("RAW file has", raw["city"].nunique(), "city strings vs", txn["city"].nunique(), "real cities - raw totals are fragmented AND inflated by duplicates")

# Ex2 - save
txn.to_csv("transactions_clean.csv", index=False); print("saved")

# Ex3 - monthly spend; missing dates must be excluded EXPLICITLY and said so
def parse_mixed(s):
    for f in ("%Y-%m-%d","%d/%m/%Y","%b %d, %y"):
        try: return pd.to_datetime(s, format=f)
        except (ValueError,TypeError): continue
    return pd.NaT
txn["dt"] = txn["txn_date"].apply(parse_mixed)
txn.loc[txn["dt"]=="1970-01-01","dt"] = pd.NaT
monthly = txn.dropna(subset=["dt"]).groupby(txn["dt"].dt.to_period("M"))["amount_inr"].sum()
print(monthly.head(3))
print(f"Excluded {txn['dt'].isna().sum()} rows with unknown dates - stated, not hidden.")

## 3C

In [ ]:
clients = pd.read_csv(BASE + "client_book.csv", parse_dates=["onboard_date"])

# Ex1 - per RM
rm = clients.groupby("relationship_manager").agg(
    clients=("client_id","count"), total_aum=("aum_inr","sum"), churn=("churned","mean")).round(3)
print("Most AUM:", rm["total_aum"].idxmax(), "| Worst churn:", rm["churn"].idxmax())

# Ex2 - spend by segment, both versions reported honestly
spend = txn.groupby("customer_id")["amount_inr"].sum().rename("total_spend").reset_index()
m = clients.merge(spend, left_on="client_id", right_on="customer_id", how="left")
m["total_spend"] = m["total_spend"].fillna(0)
print(m.groupby("segment")["total_spend"].mean().round(0))
print("Excluding zero-spend clients raises every number - report the all-clients figure, mention the split.")

# Ex3 - churn rate AND counts, always together
rate = m.pivot_table(index="city", columns="risk_profile", values="churned", aggfunc="mean").round(2)
cnt  = m.pivot_table(index="city", columns="risk_profile", values="churned", aggfunc="count")
print(rate)
print("Any 'alarming' cell with count < 15 is noise, not signal:")
print(cnt)

## 3D

In [ ]:
px = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"]).set_index("date").sort_index()
px["ret"] = px["close"].pct_change()

# Ex1 - worst month vs drawdown trough
monthly = px["ret"].resample("ME").sum()
print("Worst month:", monthly.idxmin().strftime("%Y-%m"), f"{monthly.min()*100:.1f}%")
dd = px["close"]/px["close"].cummax()-1
print("Drawdown trough:", dd.idxmin().date(), "- same episode")

# Ex2 - volume by year: 2021 is all-NaN; .mean() skips NaN silently (skipna=True default)
print(px.groupby(px.index.year)["volume"].mean().round(0))
print("2021 shows NaN: mean of nothing. On partial-missing years it would AVERAGE ONLY THE PRESENT DAYS without telling you.")

# Ex3 - days more than 10% underwater
print("Days >10% below peak:", int((dd < -0.10).sum()))